In [1]:
import torch

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [3]:
# how many different characters in this text data and stoi function
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
stoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [4]:
import random

BLOCK_SIZE = 3

def build_dataset(words_list):
    X, Y = [], []

    for w in words_list:
        context = [0] * BLOCK_SIZE

        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr   = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:n2])
Xte, Yte   = build_dataset(words[n2:])

print("Train X, Y shapes:", Xtr.shape, Ytr.shape)
print("Val X, Y shapes:  ", Xval.shape, Yval.shape)
print("Test X, Y shapes: ", Xte.shape, Yte.shape)

Train X, Y shapes: torch.Size([182418, 3]) torch.Size([182418])
Val X, Y shapes:   torch.Size([22895, 3]) torch.Size([22895])
Test X, Y shapes:  torch.Size([22833, 3]) torch.Size([22833])


In [5]:
Xtr

tensor([[ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19,  9],
        ...,
        [ 9,  7, 18],
        [ 7, 18, 21],
        [18, 21, 14]])

In [6]:
# Creating lookup table
import torch.nn as nn

C = nn.Embedding(num_embeddings=27, embedding_dim=2)

C.weight.shape

torch.Size([27, 2])

In [7]:
C(Xtr[0]) # for example input of [., ., .]

tensor([[-0.2455,  0.7869],
        [-0.2455,  0.7869],
        [-0.2455,  0.7869]], grad_fn=<EmbeddingBackward0>)

In [25]:
class NameKiller(nn.Module):
    def __init__(self):
        super(NameKiller, self).__init__()

        # 1. The Embedding Matrix (C)
        # Each character (vocab_size) gets a dense vector (emb_dim)
        self.C = nn.Embedding(27, 2)

        # 2. The Hidden Layer (H, d)
        # Input: concatenated context vectors. Output: hidden features.
        self.fc1 = nn.Linear(6, 200)

        # 3. The Output Layer (U, b)
        # Maps the hidden features back to vocabulary scores (logits)
        self.fc2 = nn.Linear(200, 27)

    def forward(self, x):
        emb = self.C(x)

        # Step B: Concatenation / Flattening
        # Glues vectors end-to-end. Output: (batch_size, context_length * emb_dim)
        x_flat = emb.view(emb.size(0), -1) #

        # Step C: The Neural Pass (tanh activation)
        # h = tanh(d + Hx)
        h = torch.tanh(self.fc1(x_flat))

        # Step D: Logits (Raw scores for every char in vocab)
        # y = b + Uh
        logits = self.fc2(h)

        return logits

In [26]:
model = NameKiller()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [27]:
for epoch in range(100):
    model.train() # Tell model it is in 'training mode'

    # 2. Forward pass on Training Data
    logits = model(Xtr)
    loss = criterion(logits, Ytr)

    # 3. Backward pass (The Calculus part)
    optimizer.zero_grad() # Reset old gradients
    loss.backward()       # Compute gradients (slopes)
    optimizer.step()      # Update Matrix C and weights

    # --- VALIDATION (Check progress) ---
    model.eval() # Tell model to stop training behaviors
    with torch.no_grad(): # Don't calculate gradients (saves memory)
        val_logits = model(Xval)
        val_loss = criterion(val_logits, Yval)
        print(f"Epoch {epoch}: Train Loss {loss.item():.4f}, Val Loss {val_loss.item():.4f}")

# --- FINAL TEST (The Final Exam) ---
# Run this ONLY ONCE after you are totally finished with training
model.eval()
with torch.no_grad():
    test_loss = criterion(model(Xte), Yte)
    print(f"FINAL PERFORMANCE ON TEST DATA: {test_loss.item():.4f}")

Epoch 0: Train Loss 3.3061, Val Loss 3.2706
Epoch 1: Train Loss 3.2691, Val Loss 3.2353
Epoch 2: Train Loss 3.2336, Val Loss 3.2017
Epoch 3: Train Loss 3.1998, Val Loss 3.1697
Epoch 4: Train Loss 3.1677, Val Loss 3.1393
Epoch 5: Train Loss 3.1373, Val Loss 3.1106
Epoch 6: Train Loss 3.1086, Val Loss 3.0836
Epoch 7: Train Loss 3.0815, Val Loss 3.0583
Epoch 8: Train Loss 3.0561, Val Loss 3.0346
Epoch 9: Train Loss 3.0324, Val Loss 3.0125
Epoch 10: Train Loss 3.0103, Val Loss 2.9921
Epoch 11: Train Loss 2.9898, Val Loss 2.9732
Epoch 12: Train Loss 2.9709, Val Loss 2.9558
Epoch 13: Train Loss 2.9536, Val Loss 2.9400
Epoch 14: Train Loss 2.9377, Val Loss 2.9255
Epoch 15: Train Loss 2.9231, Val Loss 2.9123
Epoch 16: Train Loss 2.9099, Val Loss 2.9003
Epoch 17: Train Loss 2.8979, Val Loss 2.8894
Epoch 18: Train Loss 2.8870, Val Loss 2.8796
Epoch 19: Train Loss 2.8771, Val Loss 2.8707
Epoch 20: Train Loss 2.8682, Val Loss 2.8626
Epoch 21: Train Loss 2.8600, Val Loss 2.8553
Epoch 22: Train Loss

In [45]:
import torch.nn.functional as F

def generate_name():
    g = torch.Generator().manual_seed(2147483647 + random.randint(0, 10_000))


    out = []
    context = [0] * 3

    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break


    return ''.join(itos[i] for i in out)


generate_name()


'lan.'